In [1]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch import Tensor
from torch_geometric.data import Data
from src.utils.config import PHYLO_DIST_MATRIX, LABELS_RM_METABOLITES, SIMPLE_GNN_XP_RESULTS, MODEL_CONFIGS_DIR
from collections import Counter
import torch.nn as nn
from sklearn.model_selection import train_test_split
from src.models.SimpleGNN import SimpleGNN
from torch.utils.tensorboard import SummaryWriter
import yaml
import datetime

/Users/encordsf/Desktop/phylo-gnn/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
labels = pd.read_csv(LABELS_RM_METABOLITES)
dist_matrix = np.load(PHYLO_DIST_MATRIX)

print(f"Distance matrix shape: {dist_matrix.shape}")
print(dist_matrix)
labels

Distance matrix shape: (4700, 4700)
[[0.         1.76652698 2.69671511 ... 1.6910915  1.7410793  2.37992472]
 [1.76652698 0.         2.98809765 ... 0.48802515 1.89713616 2.67130726]
 [2.69671511 2.98809765 0.         ... 2.91266216 2.96264997 2.84096409]
 ...
 [1.6910915  0.48802515 2.91266216 ... 0.         1.82170067 2.59587177]
 [1.7410793  1.89713616 2.96264997 ... 1.82170067 0.         2.64585958]
 [2.37992472 2.67130726 2.84096409 ... 2.59587177 2.64585958 0.        ]]


,genome_id,b12,heme,folate,biotin,type1,type3,rm
0,MGYG000000001,1,0,1,0,1,0,1
1,MGYG000000002,1,0,1,0,1,0,1
2,MGYG000000003,0,0,1,0,1,0,1
3,MGYG000000004,1,0,0,0,1,1,1
4,MGYG000000005,1,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...
4695,MGYG000004901,1,0,0,1,0,0,0
4696,MGYG000004902,1,0,0,0,1,0,1
4697,MGYG000004903,1,0,0,0,1,0,1
4698,MGYG000004904,0,0,0,0,0,0,0


In [3]:
print(f"Min Distance: {dist_matrix.min()}")
print(f"Max Distance: {dist_matrix.max()}")
print(f"Median Distance: {np.median(dist_matrix)}")
print(f"Labels Columns: {labels.columns}")
print(labels.sum())

Min Distance: 0.0
Max Distance: 4.8759619045
Median Distance: 2.5139894744499998
Labels Columns: Index(['genome_id', 'b12', 'heme', 'folate', 'biotin', 'type1', 'type3', 'rm'], dtype='object')
genome_id    MGYG000000001MGYG000000002MGYG000000003MGYG000...
b12                                                       1552
heme                                                       528
folate                                                    1771
biotin                                                     790
type1                                                     1609
type3                                                      838
rm                                                        2340
dtype: object


In [4]:
with open(MODEL_CONFIGS_DIR/ "SimpleGNN" / "default.yaml") as f:
    cfg = yaml.safe_load(f)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [5]:
indices = np.arange(len(labels))
print(f"Total nodes: {len(indices)}")

train_idx, temp = train_test_split(
    indices,
    shuffle=True,
    test_size=0.4,
    random_state=19)

val_idx, test_idx = train_test_split(temp, test_size=0.5, random_state=19)
print(f"Train indices: {len(train_idx)} ({len(train_idx)/len(indices)*100:.1f}%)")
print(f"Val indices: {len(val_idx)} ({len(val_idx)/len(indices)*100:.1f}%)")
print(f"Test indices: {len(test_idx)} ({len(test_idx)/len(indices)*100:.1f}%)")

Total nodes: 4700
Train indices: 2820 (60.0%)
Val indices: 940 (20.0%)
Test indices: 940 (20.0%)


In [6]:
k = cfg['k'] #Nearest neighbors

def get_knn_edges(matrix, k:int):
    edges = []
    distances = []
    for i in range(len(matrix)):
        sorted_indices = np.argsort(matrix[i])
        top_k_indices = sorted_indices[1:k+1]
        for index in list(top_k_indices):
            edges.append([i,index])
            distances.append(matrix[i][index])
    return edges, distances

edges, distances = get_knn_edges(dist_matrix,k)

In [7]:
print(f"Number of edges: {len(edges)}")
sources = [edge[0] for edge in edges]
edge_counts = Counter(sources)
print(f"Each species has {edge_counts[0]} edges (should be {k})")
for edge in edges[:100]:
    assert edge[0]!=edge[1], "Found self-loop"


Number of edges: 47000
Each species has 10 edges (should be 10)


In [8]:
edges = np.array(edges)
# print(edges[:10])
edges_transpose = np.transpose(edges)
# print(edges_transpose[:10])
edges_tensor = torch.tensor(edges_transpose)
# print(edges_tensor[:10])

In [9]:
metabolite_columns = ['b12', 'heme', 'folate', 'biotin']
metabolite_labels = labels[metabolite_columns].copy()
numpy_labels = metabolite_labels.values

x = torch.tensor(numpy_labels, dtype=torch.float)
print(x)

y = x.clone()

data = Data(x=x, edge_index=edges_tensor, y=y)
print(f"Nodes: {data.num_nodes}")
print(f"Edges: {data.num_edges}")
print(f"Node Features: {data.num_node_features}")
print(f"Nodes Features Shape: {data.x.shape}")
print(f"Edges Index Shape: {data.edge_index.shape}")

tensor([[1., 0., 1., 0.],
        [1., 0., 1., 0.],
        [0., 0., 1., 0.],
        ...,
        [1., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
Nodes: 4700
Edges: 47000
Node Features: 4
Nodes Features Shape: torch.Size([4700, 4])
Edges Index Shape: torch.Size([2, 47000])


In [10]:
train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
val_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

mask_to_hide = val_mask | test_mask
data.x[mask_to_hide] = 0

# Move all data tensors to device
data.x          = data.x.to(device)
data.y          = data.y.to(device)
data.edge_index = data.edge_index.to(device)
data.train_mask = data.train_mask.to(device)
data.val_mask   = data.val_mask.to(device)
data.test_mask  = data.test_mask.to(device)

print(f"\nSample train node features: {data.x[train_idx[1]]}")
print(f"Sample test node features: {data.x[test_idx[1]]}")


Sample train node features: tensor([1., 0., 1., 1.], device='mps:0')
Sample test node features: tensor([0., 0., 0., 0.], device='mps:0')


In [11]:
from sklearn.metrics import f1_score, roc_auc_score

def evaluate_model(model, data, mask, epoch, metadata_headers, labels_str, edge_weight, criterion, writer):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index, data.edge_weight)

    val_loss = criterion(out[mask], data.y[mask]).item()
    probs = torch.sigmoid(out[mask]).cpu().numpy()
    preds = (probs > 0.5).astype(int)
    true  = data.y[mask].cpu().numpy().astype(int)

    results = {
        "val_loss": val_loss
    }

    writer.add_scalar("Loss/val", val_loss, epoch)
    for i, trait in enumerate(metabolite_columns):
        trait_auc = roc_auc_score(true[:, i], probs[:, i])
        trait_f1 = f1_score(true[:,i], preds[:,i], zero_division=0)
        trait_name_f1 = f"F1/{trait}"
        trait_name_auc = f"AUROC/{trait}"
        writer.add_scalar(trait_name_f1, trait_f1, epoch)
        writer.add_scalar(trait_name_auc,trait_auc, epoch)

        results[trait_name_f1] = trait_f1
        results[trait_name_auc] = trait_auc

    if epoch % 25 == 0:
        embedding = model.convs[0](data.x, data.edge_index, data.edge_weight).cpu()
        writer.add_embedding(embedding, metadata=labels_str,
                             metadata_header=metadata_headers, global_step=epoch,
                             tag=f"epoch_{epoch}")
    return results

In [12]:
lr = cfg['lr']
epochs = cfg['epochs']
num_layers = cfg['num_layers']
p_dropout = cfg['p_dropout']
today = datetime.datetime.today()
hidden_dim = cfg['hidden_dim']
edge_weight_flag = cfg['edge_weight']
pos_weight_flag = cfg['pos_weight']

metadata_headers = metabolite_columns + ["genome_id"]
labels_str = labels[metadata_headers].astype(str).values.tolist()

pos_counts = torch.tensor(labels[metabolite_columns].sum().values, dtype=torch.float)
neg_counts = len(labels) - pos_counts
pos_weight = (neg_counts / pos_counts).to(device) if pos_weight_flag else None

if edge_weight_flag:
    data.edge_weight = torch.tensor([1/d for d in distances], dtype=torch.float).to(device)
else:
    data.edge_weight = None

In [15]:
def train_run(k:int=k,
              hidden_dim:int=hidden_dim,
              lr: float=lr,
              num_layers: int=num_layers,
              p_dropout: float=p_dropout,
              seed: int = None          # None = no seed prefix (grid search runs)
              ):

    if seed is not None:
        torch.manual_seed(seed)

    model = SimpleGNN(input_dim=cfg['input_dim'],
                      hidden_dim=hidden_dim,
                      output_dim=cfg['output_dim'],
                      num_layers=num_layers,
                      p_dropout=p_dropout).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses = [], []
    auroc_lists = {trait: [] for trait in metabolite_columns}
    f1_lists    = {trait: [] for trait in metabolite_columns}

    seed_tag = f"-seed{seed}" if seed is not None else ""
    run_name = f"{today.strftime('%Y-%m-%d_%H%M')}-k{k}-h{hidden_dim}-lr{lr}-l{num_layers}-dp{p_dropout}{seed_tag}"
    writer = SummaryWriter(SIMPLE_GNN_XP_RESULTS / run_name)
    writer.add_graph(model=model, input_to_model=(data.x, data.edge_index))

    for epoch in range(epochs):
        model.train()
        out = model(data.x, data.edge_index, data.edge_weight)
        optimizer.zero_grad()
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        writer.add_scalar("Loss/train", loss.item(), epoch)
        train_losses.append(loss.item())

        eval_results = evaluate_model(model, data, data.val_mask, epoch, metadata_headers,
                                      labels_str, data.edge_weight, criterion, writer)
        val_losses.append(eval_results['val_loss'])
        for trait in metabolite_columns:
            auroc_lists[trait].append(eval_results[f"AUROC/{trait}"])
            f1_lists[trait].append(eval_results[f"F1/{trait}"])

        if epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | Train Loss: {loss.item():.4f} | Val Loss: {eval_results['val_loss']:.4f}")

    # Close main writer BEFORE add_hparams — writing hparams to the same writer
    # causes TensorBoard to hide scalars in the Scalars tab (known TB bug).
    # Fix: separate writer pointing to a subdirectory so event files don't collide.
    writer.close()

    # Save model weights for every run so the best can be loaded later
    save_path = SIMPLE_GNN_XP_RESULTS / run_name / "model.pt"
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")

    best_aurocs = {trait: max(auroc_lists[trait]) for trait in metabolite_columns}
    best_f1s    = {trait: max(f1_lists[trait])    for trait in metabolite_columns}

    # Separate hparam writer to avoid the add_hparams/add_scalar collision
    hparam_writer = SummaryWriter(SIMPLE_GNN_XP_RESULTS / run_name / "hparams")
    hparam_writer.add_hparams(
        {'k': k, 'hidden_dim': hidden_dim, 'lr': lr, 'num_layers': num_layers,
         'edge_weight': cfg['edge_weight'], 'pos_weight': cfg['pos_weight']},
        {'best_train_loss': min(train_losses),
         'best_val_loss':   min(val_losses),
         'auroc_b12':       best_aurocs['b12'],
         'auroc_heme':      best_aurocs['heme'],
         'auroc_folate':    best_aurocs['folate'],
         'auroc_biotin':    best_aurocs['biotin'],
         'auroc_macro':     sum(best_aurocs.values()) / len(metabolite_columns)}
    )
    hparam_writer.close()

In [14]:
from itertools import product

param_grid = {
    # 'k':          [5, 10, 20],     # TODO: Implement Graph reconstruction for all k-values
    'hidden_dim': [32, 64, 128],   # small / current / large representation
    'num_layers': [2, 3],          # current / max before over-smoothing risk
    'lr':         [0.01, 0.001],   # 10x higher / current
    'p_dropout':  [0.3, 0.5],      # lighter / current regularisation
}

GRID_SEARCH = cfg['grid_search']

if GRID_SEARCH:
    for hidden_dim, num_layers, lr, p_dropout in product(*param_grid.values()):
        train_run(k=k, hidden_dim=hidden_dim, num_layers=num_layers, lr=lr, p_dropout=p_dropout)
else:
    train_run()

Epoch 000 | Train Loss: 1.0846 | Val Loss: 1.0746
Epoch 010 | Train Loss: 0.9356 | Val Loss: 0.9717
Epoch 020 | Train Loss: 0.8208 | Val Loss: 0.8847
Epoch 030 | Train Loss: 0.7091 | Val Loss: 0.7986
Epoch 040 | Train Loss: 0.6118 | Val Loss: 0.7285
Epoch 050 | Train Loss: 0.5296 | Val Loss: 0.6756
Epoch 060 | Train Loss: 0.4682 | Val Loss: 0.6436
Epoch 070 | Train Loss: 0.4288 | Val Loss: 0.6254
Epoch 080 | Train Loss: 0.4081 | Val Loss: 0.6102
Epoch 090 | Train Loss: 0.3902 | Val Loss: 0.6030
Epoch 100 | Train Loss: 0.3723 | Val Loss: 0.5877
Epoch 110 | Train Loss: 0.3514 | Val Loss: 0.5824
Epoch 120 | Train Loss: 0.3312 | Val Loss: 0.5796
Epoch 130 | Train Loss: 0.3338 | Val Loss: 0.5691
Epoch 140 | Train Loss: 0.3273 | Val Loss: 0.5684
Epoch 150 | Train Loss: 0.3200 | Val Loss: 0.5709
Epoch 160 | Train Loss: 0.3175 | Val Loss: 0.5683
Epoch 170 | Train Loss: 0.3180 | Val Loss: 0.5644
Epoch 180 | Train Loss: 0.3072 | Val Loss: 0.5666
Epoch 190 | Train Loss: 0.3003 | Val Loss: 0.5675


In [18]:
# After grid search: re-run best config with multiple seeds to check result stability.
# Fill in best hyperparams found from TensorBoard HParams tab, then run this cell.

BEST_CONFIG = {
    'hidden_dim': 64,    # <-- update from TensorBoard HParams tab
    'num_layers': 3,
    'lr':         0.01,
    'p_dropout':  0.5,
}
SEEDS = [0, 1, 2]

for seed in SEEDS:
    train_run(**BEST_CONFIG, seed=seed)

Epoch 000 | Train Loss: 1.0345 | Val Loss: 0.9977
Epoch 010 | Train Loss: 0.6520 | Val Loss: 0.6702
Epoch 020 | Train Loss: 0.4661 | Val Loss: 0.5237
Epoch 030 | Train Loss: 0.4293 | Val Loss: 0.5177
Epoch 040 | Train Loss: 0.4010 | Val Loss: 0.4947
Epoch 050 | Train Loss: 0.3813 | Val Loss: 0.4934
Epoch 060 | Train Loss: 0.3748 | Val Loss: 0.4903
Epoch 070 | Train Loss: 0.3735 | Val Loss: 0.4901
Epoch 080 | Train Loss: 0.3465 | Val Loss: 0.4901
Epoch 090 | Train Loss: 0.3607 | Val Loss: 0.4974
Epoch 100 | Train Loss: 0.3485 | Val Loss: 0.4863
Epoch 110 | Train Loss: 0.3488 | Val Loss: 0.4919
Epoch 120 | Train Loss: 0.3480 | Val Loss: 0.4909
Epoch 130 | Train Loss: 0.3439 | Val Loss: 0.4910
Epoch 140 | Train Loss: 0.3392 | Val Loss: 0.4953
Epoch 150 | Train Loss: 0.3467 | Val Loss: 0.4934
Epoch 160 | Train Loss: 0.3321 | Val Loss: 0.4968
Epoch 170 | Train Loss: 0.3379 | Val Loss: 0.4998
Epoch 180 | Train Loss: 0.3385 | Val Loss: 0.4881
Epoch 190 | Train Loss: 0.3295 | Val Loss: 0.4999
